In [ ]:
%load_ext autoreload
%autoreload 2

## Create train_data.json

In [ ]:
import os
import json
import pycolmap
from tqdm import tqdm
train_data = {}
path = "data"
scenes = sorted(os.listdir(path))
print(f"Found {len(scenes)} scenes.")

for scene in tqdm(scenes):
    if scene.startswith('.'):
        continue
    # print(f"Processing {scene}...")
    model_path = os.path.join(path, scene, "colmap/sparse/0")
    model = pycolmap.Reconstruction(model_path)
    cameras_data = {}
    for camera in model.cameras.values():
        camera_id = camera.camera_id
        K = camera.params
        if camera.model.name == "SIMPLE_RADIAL":
            K = [[K[0], 0, K[2]], [0, K[0], K[3]], [0, 0, 1]]
        elif camera.model.name == "PINHOLE":
            K = [[K[0], 0, K[2]], [0, K[1], K[3]], [0, 0, 1]]
        else:
            raise NotImplementedError(f"Camera model {camera.model} not supported")
        cameras_data[camera_id] = {
            "K": K,
            "width": camera.width,
            "height": camera.height
        }
    images_data = {}
    for image in model.images.values():
        image_name = image.name
        P = image.cam_from_world.matrix()
        images_data[image_name] = {
            "P": P.tolist(),
            "K_id": image.camera_id
        }
    
    train_data[scene] = {
        "cameras": cameras_data,
        "images": images_data
    }

with open("train_data.json", "w") as f:
    json.dump(train_data, f, indent=4)

## Visual data

In [ ]:
import random
from dataset_loader import TerraSkyDataset

dataset = TerraSkyDataset(
    verbose=True, 
    only_mixed=False, 
    )

In [ ]:
import random
from mylib.plot import plot_imgs, plot_imgs_and_kpts

pair_id = random.randint(0, len(dataset)-1)
out = dataset[pair_id]
img0, img1 = out['img0'].permute(1, 2, 0), out['img1'].permute(1, 2, 0)
P0, P1 = out['P0'], out['P1']
K0, K1 = out['K0'], out['K1']
Z0, Z1 = out['depth0'], out['depth1']

plot_imgs([img0, img1, Z0, Z1], titles=['img0', 'img1', 'depth0', 'depth1'])

In [ ]:
from dataset_viewer_helpers import compute_121_reprojection

data = {
    'P0': P0[None], 'P1': P1[None],
    'K0': K0[None],  'K1': K1[None],
    'depth0': Z0[None], 'depth1': Z1[None]
}
kpts1, kpts2, tot, distances = compute_121_reprojection(
    data,
    img0.permute(2,0,1), img1.permute(2,0,1), 
    verbose=False, reprojection_error=5.0, border=0, sampling_factor=1)

print(f"Consistent points: {len(kpts1):,}, {len(kpts2):,} out of {tot:,} ({100*len(kpts1)/tot:.4f}%)")

plot_imgs_and_kpts(img0*255//1, img1*255//1, kpts1, kpts2, sample_points=5_000, index=False, matches=False) 